<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [환경 설정] 필수 라이브러리 설치
# faster-whisper: 음성 인식(STT) 모델
# rapidfuzz: 문자열 유사도 계산 (Levenshtein 거리 등) - 정답과 예측값 비교용
# g2pk: 한국어 발음 변환기 (Grapheme-to-Phoneme)
# konlpy, python-mecab-ko: 한국어 형태소 분석기 (단어 단위 오류율 측정용)
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko torch

In [2]:
# 1. 시스템 및 유틸리티 라이브러리
import gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
import importlib

# 2. 자연어 처리 및 문자열 매칭 라이브러리
from g2pk import G2p  # 발음 변환
from rapidfuzz.distance import Levenshtein # 편집 거리 계산
from rapidfuzz import process, fuzz # 문자열 유사도 처리
from mecab import MeCab # 형태소 분석

# 3. AI 모델 라이브러리
from faster_whisper import WhisperModel # Whisper ASR 모델

In [3]:
# # [Google Colab 전용] 구글 드라이브 마운트
# # 로컬 환경이 아닌 Colab에서 실행 시 주석을 해제하여 드라이브를 연결합니다.
from google.colab import drive
drive.mount('/content/drive')

# [개발 편의 설정] 모듈 자동 리로드
# 외부 .py 파일을 수정했을 때, 커널(세션)을 껐다 켜지 않아도
# 자동으로 변경 사항이 반영되도록 설정합니다.
# %load_ext autoreload
# %autoreload 2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# [경로 설정]
# 프로젝트 파일들이 위치한 경로를 설정하거나 이동합니다. (현재는 주석 처리됨)
BASE_PATH = "/content/drive/MyDrive/results/"
%cd /content/drive/MyDrive/results/

# [사용자 정의 모듈 Import]
# config.settings: 실험에 필요한 상수 및 설정값(경로, 모델명 등) 로드
from config.settings import *

# utils: 텍스트 정규화, 데이터 로더, 성능 평가 지표(CER/WER) 계산 도구
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns

# core: ASR 엔진, 핫워드 편향(Bias) 관리, 후처리 로직
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords
from utils.exporter import *

/content/drive/MyDrive/results


In [5]:
# [메모리 관리] GPU 메모리 누수를 방지하기 위한 초기화
# import gc
# gc.collect()
# torch.cuda.empty_cache()

# ==============================================================================
# [실험 설정 및 초기화]
# ==============================================================================
import os
import json # replog json dump를 위해 필요
from config.settings import RESULTS_DIR
import random

# 실험의 재현성을 위해 랜덤 시드 고정
random.seed(42)

# 1. 결과 저장 디렉토리 생성
os.makedirs(RESULTS_DIR, exist_ok=True)

# 2. 결과 파일명 설정 (타임스탬프 포함)
now = time.gmtime(time.time()+(9*3600)) # 한국 시간(KST)
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_comparison.csv"   # 비교 실험 결과 저장

# 3. 핵심 객체 초기화
normalizer = TextNormalizer()   # 텍스트 정규화기
mecab = MeCab()                 # 형태소 분석기
bias_mgr = BiasManager(BIAS_PATH) # 핫워드 가중치 관리자
transcripts = load_transcripts(TRANSCRIPTS_PATH) # 정답지 로드
files = glob.glob(os.path.join(AUDIO_FOLDER, "*.mp4"), recursive=True) # 오디오 파일 목록


# Baseline용 ASR 모델 로드 (initial_prompt를 None으로 설정하여 아무런 힌트도 주지 않음)
asr_baseline = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE, initial_prompt="")
rows_baseline = []

# 데이터 편향 방지를 위해 파일 순서 섞기 (Baseline은 1회만 수행하므로 여기서 한 번 섞음)
random.shuffle(files)


# Proposed용 ASR 모델 로드 (한국어 특화 Prompt 적용)
asr_proposed = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE, initial_prompt=KOREAN_ONLY_PROMPT)
rows_proposed = []


# 기존의 중첩 반복문 (Grid Search & Iterative Learning)
for top_k in HOTWORD_TOPK_SWEEP:
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP:
        for bias_weight_update_cnt in BIAS_WEIGHT_UPDATE_ITERATION_SWEEP:

            # (옵션) 학습 리셋: 새로운 실험 사이클마다 편향 리스트 초기화
            if RESET_BIASING_LIST:
                bias_mgr.reset_biasing_list(BIAS_PATH)

            # 피드백 루프 (Iterative Learning)
            for repeat in range(bias_weight_update_cnt):
                # 핫워드 추출 (가중치 기반)
                current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)

                for pp_on in POSTPROCESS_SWEEP:
                    pp_str= "ON" if pp_on ==1 else "OFF"
                    print(f"\n[Proposed] Iteration: {repeat+1}/{bias_weight_update_cnt} | Hotwords: {len(current_hotwords)} | PP: {pp_str}")

                    # 데이터 순서 섞기
                    random.shuffle(files)

                    for audio_path in files[:AUDIO_FILE_MAX]:
                        fname = os.path.basename(audio_path)
                        meta = transcripts.get(fname, {"text": "", "entities": []})
                        # ==============================================================================
                        # [PHASE 1] Baseline 실험 (No Prompt, No Hotwords)
                        # 목적: 핫워드 바이어싱이나 프롬프트가 없을 때의 순수 Whisper 성능 측정
                        # ==============================================================================

                        # 1) ASR 추론: 핫워드 없이(hotwords=None) 순수 성능 측정
                        hyp_raw = asr_baseline.transcribe(audio_path, "ko", ASR_BEAM, hotwords=None)

                        # 2) 후처리: Baseline이므로 후처리(교정) 로직을 적용하지 않음
                        hyp_final = normalizer.normalize(hyp_raw)
                        replog = []

                        # 3) 성능 평가 (CER, WER, PN_Recall 등)
                        # 정답 텍스트 정규화
                        ref_final = normalizer.normalize(meta["text"], False)

                        # 고유명사(PN) 성능 평가
                        pn_recall, wrong_pn_char_cnt, pn_char_cnt, pn_cer, wrong_pn_morph_cnt, pn_morph_cnt, pn_wer, hyp_ents, soft_missed_ents, _ = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH, hotwords=[])

                        # 전체 문장 성능 평가
                        cer, wrong_char_cnt, char_cnt = calculate_cer(ref_final, hyp_final, normalizer)
                        wer, wrong_morph_cnt, morph_cnt, ref_morphs, hyp_morphs = calculate_wer(ref_final, hyp_final, normalizer, mecab, meta["entities"], hyp_ents)

                        # 로그 출력
                        print(f"[Baseline] {fname} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer:.4f} | pn_wer={pn_wer:.4f}")

                        # 결과 리스트에 추가 (experiment_type='baseline'으로 태깅)
                        rows_baseline.append({
                            "experiment_type": "baseline", # 실험 타입 구분자
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            # Baseline은 아래 옵션들이 없으므로 0 또는 None 처리
                            "top_k": 0, "postprocess_on": 0, "hotwords_strategy": "none", "hotwords_hit":"",
                            "bias_weight_update_cnt": repeat+1, "hotwords": "None",

                            # 성능 지표
                            "cer": f"{cer:.4f}", "wer": f"{wer:.4f}",
                            "pn_cer": None if pn_cer is None else float(f"{pn_cer:.4f}"),
                            "pn_wer": None if pn_wer is None else float(f"{pn_wer:.4f}"),
                            "pn_recall": None if pn_recall is None else float(f"{pn_recall:.4f}"),

                            # 텍스트 데이터
                            "ref_raw": meta["text"], "hyp_raw": hyp_raw,
                            "ref_final": ref_final, "hyp_final": hyp_final,
                            "ref_pn": meta.get("entities", []), "hyp_pn": hyp_ents,
                            "soft_missed_pn": soft_missed_ents,
                            "replog": "[]", # 교정 로그 없음

                            # 상세 카운트 (Summary 집계용)
                            "wrong_char_cnt": wrong_char_cnt, "char_cnt": char_cnt,
                            "wrong_morph_cnt": wrong_morph_cnt, "morph_cnt": morph_cnt,
                            "wrong_pn_char_cnt": wrong_pn_char_cnt, "pn_char_cnt": pn_char_cnt,
                            "wrong_pn_morph_cnt": wrong_pn_morph_cnt, "pn_morph_cnt": pn_morph_cnt
                        })



# ==============================================================================
# [PHASE 2] Proposed 실험 (With Prompt, With Hotwords)
# 목적: 적응형 바이어싱(Adaptive Biasing)과 프롬프트를 적용했을 때의 성능 향상 측정
# ==============================================================================

                        # 1) ASR 추론 (핫워드 적용)
                        hyp_raw = asr_proposed.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)

                        # 2) 후처리 (Post-processing) 적용 여부 확인
                        if pp_on:
                            hyp_final, replog = postprocess_with_hotwords(
                                hyp_raw, current_hotwords, normalizer,
                                gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                            )
                        else:
                            hyp_final, replog = normalizer.normalize(hyp_raw,True),[]

                        # 3) 성능 평가
                        ref_final = normalizer.normalize(meta["text"], False)

                        pn_recall, wrong_pn_char_cnt, pn_char_cnt, pn_cer, wrong_pn_morph_cnt, pn_morph_cnt, pn_wer, hyp_ents, soft_missed_ents, hotwords_hit = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH, hotwords=current_hotwords)

                        cer, wrong_char_cnt, char_cnt = calculate_cer(ref_final, hyp_final, normalizer)
                        wer, wrong_morph_cnt, morph_cnt, ref_morphs, hyp_morphs = calculate_wer(ref_final, hyp_final, normalizer, mecab, meta["entities"], hyp_ents)

                        # [학습 단계] 미인식된 고유명사를 Bias Manager에 전달하여 가중치 증가
                        bias_mgr.add_miss(soft_missed_ents)

                        # 로그 출력
                        print(f"[Proposed] {fname} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer:.4f} | pn_wer={pn_wer:.4f}")

                        # 결과 리스트에 추가 (experiment_type='proposed'로 태깅)
                        rows_proposed.append({
                            "experiment_type": "proposed", # 실험 타입 구분자
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            "top_k": top_k,
                            "postprocess_on": int(pp_on),
                            "hotwords_strategy": "random" if hotwords_strategy == 1 else "hybrid",
                            "bias_weight_update_cnt": repeat+1,
                            "hotwords": current_hotwords,
                            "hotwords_hit":hotwords_hit,

                            "cer": f"{cer:.4f}", "wer": f"{wer:.4f}",
                            "pn_cer": None if pn_cer is None else float(f"{pn_cer:.4f}"),
                            "pn_wer": None if pn_wer is None else float(f"{pn_wer:.4f}"),
                            "pn_recall": None if pn_recall is None else float(f"{pn_recall:.4f}"),

                            "ref_raw": meta["text"], "hyp_raw": hyp_raw,
                            "ref_final": ref_final, "hyp_final": hyp_final,
                            "ref_pn": meta.get("entities", []), "hyp_pn": hyp_ents,
                            "soft_missed_pn": soft_missed_ents,
                            "replog": json.dumps(replog, ensure_ascii=False), # 교정 로그 저장

                            "wrong_char_cnt": wrong_char_cnt, "char_cnt": char_cnt,
                            "wrong_morph_cnt": wrong_morph_cnt, "morph_cnt": morph_cnt,
                            "wrong_pn_char_cnt": wrong_pn_char_cnt, "pn_char_cnt": pn_char_cnt,
                            "wrong_pn_morph_cnt": wrong_pn_morph_cnt, "pn_morph_cnt": pn_morph_cnt
                        })

                # 한 이터레이션 종료 후 학습된 가중치를 파일에 영구 반영
                bias_mgr.finalize(repeat)

# ==============================================================================
# [결과 통합 및 저장]
# ==============================================================================
# Baseline과 Proposed 결과를 하나의 리스트로 합침 (Proposed 결과를 상단에 배치)
all_rows = rows_proposed + rows_baseline

# CSV 파일로 저장 (Summary 포함)
save_results_with_summary(all_rows, OUT_ROWS)

print(f"\n[DONE] All experiments finished. Results saved to {OUT_ROWS}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[SUCCESS] /content/drive/MyDrive/results/lists/biasing_list.json 데이터가 모두 0으로 초기화되었습니다.

[Proposed] Iteration: 1/2 | Hotwords: 0 | PP: OFF
[Baseline] 163.mp4 | cer=1.2727 | wer=1.3333 | pn_cer=0.6875 | pn_wer=0.6667
[Proposed] 163.mp4 | cer=0.1364 | wer=0.5000 | pn_cer=0.3750 | pn_wer=0.6667
[Baseline] 064.mp4 | cer=0.0000 | wer=0.0000 | pn_cer=0.0000 | pn_wer=0.0000
[Proposed] 064.mp4 | cer=0.0000 | wer=0.0000 | pn_cer=0.0000 | pn_wer=0.0000
[Baseline] 319.mp4 | cer=0.0500 | wer=0.1429 | pn_cer=0.1111 | pn_wer=0.3333
[Proposed] 319.mp4 | cer=0.0500 | wer=0.4286 | pn_cer=0.1111 | pn_wer=0.3333
[Baseline] 356.mp4 | cer=0.0000 | wer=0.0000 | pn_cer=0.0000 | pn_wer=0.0000
[Proposed] 356.mp4 | cer=0.0000 | wer=0.1667 | pn_cer=0.0000 | pn_wer=0.0000
[Baseline] 053.mp4 | cer=0.1034 | wer=0.1875 | pn_cer=0.0000 | pn_wer=0.0000
[Proposed] 053.mp4 | cer=0.1034 | wer=0.1875 | pn_cer=0.0000 | pn_wer=0.0000

[LEARNING] 1회차 가중치 누적 저장 완료. (반복횟수=1)

[Proposed] Iteration: 2/2 | Hotwords: 3 | PP: OFF
[B